## Cell 1 — Imports and Dependencies

In [4]:
import hashlib
import time
import json
import copy

print("Libraries loaded successfully.")


Libraries loaded successfully.


## Cell 2 — Block Class Structure

In [5]:
class Block:
    """
    Represents a single block in the blockchain.
    Each block stores: index, previous hash, timestamp, data, nonce, and its own hash.
    """

    def __init__(self, index, previous_hash, timestamp, data, nonce=0):
        self.index         = index
        self.previous_hash = previous_hash
        self.timestamp     = timestamp
        self.data          = data
        self.nonce         = nonce
        self.hash          = self.compute_hash()

    def compute_hash(self):
        """Serialises block fields and returns a SHA-256 hash string."""
        block_content = json.dumps({
            "index":         self.index,
            "previous_hash": self.previous_hash,
            "timestamp":     self.timestamp,
            "data":          self.data,
            "nonce":         self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_content.encode()).hexdigest()

    def __repr__(self):
        return (
            f"Block(index={self.index}, "
            f"hash={self.hash[:12]}..., "
            f"prev={self.previous_hash[:12]}..., "
            f"nonce={self.nonce})"
        )


# ── Quick test ──
test_block = Block(
    index=0,
    previous_hash="0" * 64,
    timestamp=time.time(),
    data="Test block"
)
print("Block created:")
print(f"  Index         : {test_block.index}")
print(f"  Data          : {test_block.data}")
print(f"  Hash          : {test_block.hash}")
print(f"  Previous Hash : {test_block.previous_hash}")


Block created:
  Index         : 0
  Data          : Test block
  Hash          : 10a9d2220eafd40e4d8a6bf64b961c69057669b36d6fe3d994ed6c9e35baaf44
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000


## Cell 3 — Genesis Block Creation

In [6]:
def create_genesis_block():
    """
    Creates the first block in the chain (Block 0).
    Has no predecessor, so previous_hash is set to a string of 64 zeros.
    """
    return Block(
        index=0,
        previous_hash="0" * 64,
        timestamp=time.time(),
        data="Genesis Block — HealthCare Innovations Patient Record System"
    )


genesis = create_genesis_block()

print("Genesis block created:")
print(f"  Index         : {genesis.index}")
print(f"  Data          : {genesis.data}")
print(f"  Timestamp     : {genesis.timestamp}")
print(f"  Previous Hash : {genesis.previous_hash}")
print(f"  Hash          : {genesis.hash}")


Genesis block created:
  Index         : 0
  Data          : Genesis Block — HealthCare Innovations Patient Record System
  Timestamp     : 1780388707.287433
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : d73e770984bb1c7e97a491cfa33ab9f3c73a07ccddd203f2b61cb92358ab070e


## Cell 4 — Proof-of-Work Mechanism

In [7]:
DIFFICULTY = 4  # Number of leading zeros required in a valid hash

def proof_of_work(block):
    """
    Iteratively increments the nonce until the block's hash
    starts with a number of zeros equal to DIFFICULTY.
    Returns the valid hash once found.
    """
    target = "0" * DIFFICULTY
    block.nonce = 0

    while not block.hash.startswith(target):
        block.nonce += 1
        block.hash = block.compute_hash()

    return block.hash


# ── Test proof-of-work on a sample block ──
sample_block = Block(
    index=1,
    previous_hash=genesis.hash,
    timestamp=time.time(),
    data={"patient_id": "P001", "action": "record_added", "provider": "Dr. Smith"}
)

print(f"Mining block {sample_block.index} with difficulty {DIFFICULTY}...")
start = time.time()
valid_hash = proof_of_work(sample_block)
elapsed = time.time() - start

print(f"  Block mined in {elapsed:.3f}s")
print(f"  Nonce found   : {sample_block.nonce}")
print(f"  Valid hash    : {valid_hash}")
print(f"  Starts with {DIFFICULTY} zeros: {valid_hash.startswith('0' * DIFFICULTY)}")


Mining block 1 with difficulty 4...
  Block mined in 1.022s
  Nonce found   : 87406
  Valid hash    : 0000d9d417e71281c329087ef94c58c2ac6efc26b647f11d7906aaa2979c721c
  Starts with 4 zeros: True


## Cell 5 — Blockchain Class and Adding New Blocks

In [8]:
class Blockchain:
    """
    Manages the chain of blocks for the patient record system.
    Handles genesis block creation, adding new blocks, and chain validation.
    """

    def __init__(self):
        self.chain      = []
        self.difficulty = DIFFICULTY
        self.chain.append(create_genesis_block())

    @property
    def last_block(self):
        return self.chain[-1]

    def add_block(self, data):
        """
        Creates a new block with the given data, runs proof-of-work,
        and appends it to the chain.
        """
        new_block = Block(
            index=len(self.chain),
            previous_hash=self.last_block.hash,
            timestamp=time.time(),
            data=data
        )
        proof_of_work(new_block)
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self):
        """
        Validates the entire chain by checking:
        - Each block's stored hash matches its recomputed hash.
        - Each block's previous_hash matches the actual hash of the prior block.
        """
        for i in range(1, len(self.chain)):
            current  = self.chain[i]
            previous = self.chain[i - 1]

            if current.hash != current.compute_hash():
                print(f"  [INVALID] Block {i} hash mismatch.")
                return False

            if current.previous_hash != previous.hash:
                print(f"  [INVALID] Block {i} previous_hash does not match Block {i-1} hash.")
                return False

        return True

    def display_chain(self):
        for block in self.chain:
            print(f"  Block {block.index} | Hash: {block.hash[:20]}... | "
                  f"Prev: {block.previous_hash[:20]}... | Nonce: {block.nonce}")


# ── Build a chain and add patient record blocks ──
hc_chain = Blockchain()

records = [
    {"patient_id": "P001", "action": "record_created",   "provider": "Dr. Smith",   "hospital": "City Hospital"},
    {"patient_id": "P002", "action": "record_created",   "provider": "Dr. Patel",   "hospital": "North Clinic"},
    {"patient_id": "P001", "action": "record_transferred","from": "Dr. Smith",       "to": "Dr. Jones"},
]

print(f"Adding {len(records)} patient record blocks...\n")
for record in records:
    added = hc_chain.add_block(record)
    print(f"  Added: Block {added.index} | Action: {record['action']} | Hash: {added.hash[:20]}...")

print(f"\nChain length : {len(hc_chain.chain)} blocks")
print(f"Chain valid  : {hc_chain.is_chain_valid()}")
print("\nFull chain:")
hc_chain.display_chain()


Adding 3 patient record blocks...

  Added: Block 1 | Action: record_created | Hash: 000049460ab5e71fb99a...
  Added: Block 2 | Action: record_created | Hash: 00006208219caa927afc...
  Added: Block 3 | Action: record_transferred | Hash: 0000294f49ada612321a...

Chain length : 4 blocks
Chain valid  : True

Full chain:
  Block 0 | Hash: caa2641ccfba1c3ebce1... | Prev: 00000000000000000000... | Nonce: 0
  Block 1 | Hash: 000049460ab5e71fb99a... | Prev: caa2641ccfba1c3ebce1... | Nonce: 140375
  Block 2 | Hash: 00006208219caa927afc... | Prev: 000049460ab5e71fb99a... | Nonce: 41362
  Block 3 | Hash: 0000294f49ada612321a... | Prev: 00006208219caa927afc... | Nonce: 7436


## Cell 6 — Consensus Mechanism (Longest Chain Rule)

In [9]:
def consensus(nodes):
    """
    Implements the longest-chain consensus rule.
    Given a list of Blockchain instances (representing network nodes),
    returns the longest valid chain found across all nodes.

    In a real distributed network, each node would broadcast its chain length
    and share its full chain on request. Here we simulate that locally.
    """
    longest_chain  = None
    max_length     = 0

    for i, node in enumerate(nodes):
        if node.is_chain_valid() and len(node.chain) > max_length:
            max_length    = len(node.chain)
            longest_chain = node.chain
            print(f"  Node {i}: valid chain, length {len(node.chain)} — current winner")
        else:
            print(f"  Node {i}: chain length {len(node.chain)} — not selected")

    return longest_chain


# ── Simulate three network nodes with different chain lengths ──

# Node A — our main chain (4 blocks including genesis)
node_a = hc_chain

# Node B — shorter chain (2 blocks)
node_b = Blockchain()
node_b.add_block({"patient_id": "P010", "action": "record_created", "provider": "Dr. Lee"})

# Node C — longer chain (6 blocks), simulating a node that processed more records
node_c = Blockchain()
extra_records = [
    {"patient_id": "P003", "action": "record_created",    "provider": "Dr. Brown"},
    {"patient_id": "P004", "action": "record_created",    "provider": "Dr. Khan"},
    {"patient_id": "P005", "action": "record_created",    "provider": "Dr. Ali"},
    {"patient_id": "P003", "action": "record_transferred","from": "Dr. Brown", "to": "Dr. Smith"},
    {"patient_id": "P006", "action": "record_created",    "provider": "Dr. Nguyen"},
]
for r in extra_records:
    node_c.add_block(r)

print("Running consensus across 3 nodes:\n")
agreed_chain = consensus([node_a, node_b, node_c])

print(f"\nConsensus result: accepted chain with {len(agreed_chain)} blocks.")
print("Accepted chain blocks:")
for block in agreed_chain:
    print(f"  Block {block.index} | Hash: {block.hash[:20]}...")


Running consensus across 3 nodes:

  Node 0: valid chain, length 4 — current winner
  Node 1: chain length 2 — not selected
  Node 2: valid chain, length 6 — current winner

Consensus result: accepted chain with 6 blocks.
Accepted chain blocks:
  Block 0 | Hash: a440f24bcd85ec5bc899...
  Block 1 | Hash: 0000db4e54c628c7183a...
  Block 2 | Hash: 000093994b1fa941b2bd...
  Block 3 | Hash: 0000dc349b65c0100778...
  Block 4 | Hash: 0000bd688cfac3589215...
  Block 5 | Hash: 0000d061fa2d98a56a41...


## Cell 7 — Full End-to-End Demo

In [11]:
print("=" * 60)
print("  HealthCare Innovations — Patient Record Blockchain Demo")
print("=" * 60)

# 1. Initialise a fresh blockchain
demo_chain = Blockchain()
print(f"\n[1] Blockchain initialised. Genesis block hash: {demo_chain.last_block.hash[:20]}...")

# 2. Add patient records
demo_records = [
    {"patient_id": "P100", "action": "record_created",    "provider": "Dr. Ahmed",  "hospital": "East Wing"},
    {"patient_id": "P101", "action": "record_created",    "provider": "Dr. Patel",  "hospital": "West Clinic"},
    {"patient_id": "P100", "action": "record_transferred","from": "Dr. Ahmed",      "to": "Dr. Patel"},
    {"patient_id": "P102", "action": "record_created",    "provider": "Dr. Okafor", "hospital": "Central Hospital"},
]

print(f"\n[2] Mining {len(demo_records)} patient record blocks:")
for record in demo_records:
    b = demo_chain.add_block(record)
    print(f"    Block {b.index} mined | Nonce: {b.nonce:<6} | Action: {record['action']}")



  HealthCare Innovations — Patient Record Blockchain Demo

[1] Blockchain initialised. Genesis block hash: d4526363e736c35f9bae...

[2] Mining 4 patient record blocks:
    Block 1 mined | Nonce: 38624  | Action: record_created
    Block 2 mined | Nonce: 247    | Action: record_created
    Block 3 mined | Nonce: 120224 | Action: record_transferred
    Block 4 mined | Nonce: 139424 | Action: record_created


In [12]:
# 3. Validate chain integrity
print(f"\n[3] Chain integrity check: {'PASSED' if demo_chain.is_chain_valid() else 'FAILED'}")





[3] Chain integrity check: PASSED


In [13]:
# 4. Tamper with a block and re-validate
print("\n[4] Simulating tampering with Block 1 data...")
demo_chain.chain[1].data = {"patient_id": "P999", "action": "TAMPERED"}
print(f"    Chain integrity after tampering: {'PASSED' if demo_chain.is_chain_valid() else 'FAILED — Tampering detected'}")




[4] Simulating tampering with Block 1 data...
  [INVALID] Block 1 hash mismatch.
    Chain integrity after tampering: FAILED — Tampering detected


In [14]:
# 5. Consensus
print("\n[5] Consensus test:")
fresh_chain = Blockchain()
for record in demo_records:
    fresh_chain.add_block(record)

winner = consensus([demo_chain, fresh_chain])
print(f"\n    Accepted chain length: {len(winner)} blocks")

print("\n" + "=" * 60)
print("  Demo complete.")
print("=" * 60)


[5] Consensus test:
  [INVALID] Block 1 hash mismatch.
  Node 0: chain length 5 — not selected
  Node 1: valid chain, length 5 — current winner

    Accepted chain length: 5 blocks

  Demo complete.
